In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import userdata

Mounted at /content/drive


In [ ]:
! pip install sentence-transformers chromadb flashrank openai tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

# Task 3.1 - Compute & Report Metrics

## Generate Full Results

Selected strategies:

- baseline - recursive_chunks
- baseline - section_chunks
- reranked - section_chunks
- hybrid - section_chunks

In [ ]:
# ============================================================
# TASK 2.1 — RETRIEVAL PIPELINE
# STRATEGIES: BASELINE COSINE, RE-RANKING, HYDE, MULTI-QUERY
# MODEL: BAAI/BGE-SMALL-EN-V1.5 | DB: CHROMADB
# ============================================================

import json
import os
import time
from typing import Any

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI
from tqdm import tqdm

# ── CONFIG ────────────────────────────────────────────────
CHROMA_PATH   = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/chromadb"  # **CHROMADB PATH ON GOOGLE DRIVE**
QUERIES_PATH  = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/queries.json"   # **PATH TO QUERIES FILE**
QRELS_PATH    = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/qrels.json"     # **PATH TO QRELS FILE**
EMBED_MODEL   = "BAAI/bge-small-en-v1.5"   # **SAME EMBEDDING MODEL USED DURING INDEXING**
OPENAI_KEY    = userdata.get('OPENAI_API_KEY')
TOP_K         = 3    # **NUMBER OF FINAL CHUNKS TO RETURN**
FETCH_K       = 20   # **CANDIDATE POOL SIZE FOR RE-RANKING**

# ── LOAD MODELS & CLIENTS ─────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)  # **BGE EMBEDDING MODEL FOR QUERY ENCODING**

print("Loading cross-encoder for re-ranking...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")  # **CROSS-ENCODER FOR RE-RANKING**

print("Connecting to ChromaDB...")
chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH,
    settings=Settings(anonymized_telemetry=False)
)  # **PERSISTENT CHROMADB CLIENT POINTING TO GOOGLE DRIVE**


Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

In [ ]:
col_recursive = chroma_client.get_collection("recursive_chunks")  # **RECURSIVE CHUNKING COLLECTION**
col_section   = chroma_client.get_collection("section_chunks")    # **SECTION-AWARE CHUNKING COLLECTION**

openai_client = OpenAI(api_key=OPENAI_KEY)  # **OPENAI CLIENT FOR HYDE AND MULTI-QUERY**

In [ ]:
# ══════════════════════════════════════════════════════════
# HELPER: EMBED A QUERY STRING
# ══════════════════════════════════════════════════════════
def embed_query(text: str) -> list[float]:
    """EMBED A SINGLE QUERY STRING USING BGE MODEL**"""
    return embedder.encode(text, normalize_embeddings=True).tolist()


# ══════════════════════════════════════════════════════════
# HELPER: PARSE RAW CHROMA RESULTS INTO CLEAN CHUNK DICTS
# ══════════════════════════════════════════════════════════
def parse_results(results: dict) -> list[dict]:
    """CONVERT CHROMADB RESULT DICT INTO LIST OF CHUNK DICTS**"""
    chunks = []
    ids        = results["ids"][0]        # **CHUNK IDS**
    documents  = results["documents"][0]  # **CHUNK TEXT**
    metadatas  = results["metadatas"][0]  # **CHUNK METADATA**
    distances  = results["distances"][0]  # **COSINE DISTANCES (LOWER = MORE SIMILAR)**

    for i, chunk_id in enumerate(ids):
        chunks.append({
            "chunk_id":  chunk_id,
            "text":      documents[i],
            "arxiv_id":  metadatas[i].get("arxiv_id", ""),
            "strategy":  metadatas[i].get("strategy", ""),
            "chunk_idx": metadatas[i].get("chunk_index", -1),
            "score":     1 - distances[i],  # **CONVERT DISTANCE TO SIMILARITY SCORE**
        })
    return chunks

In [ ]:
!pip install -q rank_bm25

In [ ]:
# ══════════════════════════════════════════════════════════
# STRATEGY 1 — BASELINE COSINE RETRIEVAL
# ══════════════════════════════════════════════════════════
def retrieve_baseline(query: str, collection, k: int = TOP_K) -> list[dict]:
    """RETRIEVE TOP-K CHUNKS VIA COSINE SIMILARITY BASELINE**"""
    query_vec = embed_query(query)  # **EMBED QUERY WITH BGE**
    results = collection.query(
        query_embeddings=[query_vec],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )  # **CHROMADB NEAREST NEIGHBOUR LOOKUP**
    return parse_results(results)


# ══════════════════════════════════════════════════════════
# STRATEGY 2 — RE-RANKING (CROSS-ENCODER)
# ══════════════════════════════════════════════════════════
def retrieve_reranked(query: str, collection, k: int = TOP_K, fetch_k: int = FETCH_K) -> list[dict]:
    """RETRIEVE FETCH_K CANDIDATES VIA COSINE THEN RE-RANK WITH CROSS-ENCODER**"""
    # STEP 1: FETCH LARGER CANDIDATE POOL **
    query_vec = embed_query(query)
    results = collection.query(
        query_embeddings=[query_vec],
        n_results=fetch_k,
        include=["documents", "metadatas", "distances"]
    )
    candidates = parse_results(results)  # **CANDIDATE CHUNKS BEFORE RE-RANKING**

    # STEP 2: SCORE EACH CANDIDATE WITH CROSS-ENCODER **
    pairs = [(query, c["text"]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs)  # **CROSS-ENCODER RELEVANCE SCORES**

    # STEP 3: ATTACH SCORES AND SORT DESCENDING **
    for i, chunk in enumerate(candidates):
        chunk["ce_score"] = float(ce_scores[i])

    reranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:k]  # **RETURN TOP-K AFTER RE-RANKING**

# ══════════════════════════════════════════════════════════
# STRATEGY 5 — HYBRID SEARCH (BM25 + DENSE + RRF)
# ══════════════════════════════════════════════════════════
from rank_bm25 import BM25Okapi
def build_bm25_index(collection) -> tuple:
    """
    BUILD BM25 INDEX FROM ALL CHUNKS IN A CHROMADB COLLECTION**
    RETURNS: (BM25 OBJECT, LIST OF ALL CHUNKS)**
    CALL ONCE PER COLLECTION AND CACHE**
    """
    print(f"Building BM25 index for {collection.name}...")

    all_data = collection.get(include=["documents", "metadatas"])

    all_chunks = []
    for i, doc in enumerate(all_data["documents"]):
        all_chunks.append({
            "chunk_id":  all_data["ids"][i],
            "text":      doc,
            "arxiv_id":  all_data["metadatas"][i].get("arxiv_id", ""),
            "strategy":  all_data["metadatas"][i].get("strategy", ""),
            "chunk_idx": all_data["metadatas"][i].get("chunk_index", -1),
            "score":     0.0,
        })

    # TOKENIZE CORPUS FOR BM25 — WHITESPACE SPLIT**
    tokenized_corpus = [chunk["text"].lower().split() for chunk in all_chunks]
    bm25 = BM25Okapi(tokenized_corpus)  # **BUILD BM25 INDEX**

    print(f"BM25 index built: {len(all_chunks)} chunks")
    return bm25, all_chunks


def reciprocal_rank_fusion(rankings: list, k: int = 60) -> list:
    """
    COMBINE RANKED LISTS USING RECIPROCAL RANK FUSION**
    RRF SCORE = SUM OF 1/(k + rank) ACROSS ALL LISTS**
    K=60 IS THE STANDARD RRF CONSTANT**
    """
    rrf_scores = {}

    for ranked_list in rankings:
        for rank, chunk_id in enumerate(ranked_list, 1):
            if chunk_id not in rrf_scores:
                rrf_scores[chunk_id] = 0.0
            rrf_scores[chunk_id] += 1.0 / (k + rank)  # **RRF FORMULA**

    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)


def retrieve_hybrid(query: str, collection,
                    k: int = TOP_K,
                    bm25_index = None,
                    all_chunks = None,
                    fetch_k: int = FETCH_K) -> list:
    """
    HYBRID RETRIEVAL: BM25 + DENSE VECTOR SEARCH VIA RRF**
    STEP 1: BM25 KEYWORD SEARCH → RANKED LIST A**
    STEP 2: COSINE VECTOR SEARCH → RANKED LIST B**
    STEP 3: RECIPROCAL RANK FUSION → MERGED RANKING**
    STEP 4: RETURN TOP-K FROM MERGED RANKING**
    """
    if bm25_index is None or all_chunks is None:
        raise ValueError("hybrid requires bm25_index and all_chunks")

    # STEP 1: BM25 KEYWORD SEARCH **
    tokenized_query  = query.lower().split()
    bm25_scores      = bm25_index.get_scores(tokenized_query)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:fetch_k]
    bm25_ranking     = [all_chunks[i]["chunk_id"] for i in bm25_top_indices]

    # STEP 2: DENSE VECTOR SEARCH **
    query_vec     = embed_query(query)
    dense_results = collection.query(
        query_embeddings=[query_vec],
        n_results=fetch_k,
        include=["documents", "metadatas", "distances"]
    )
    dense_chunks  = parse_results(dense_results)
    dense_ranking = [c["chunk_id"] for c in dense_chunks]

    # STEP 3: RECIPROCAL RANK FUSION **
    fused_ranking = reciprocal_rank_fusion([bm25_ranking, dense_ranking])

    # STEP 4: BUILD RESULT LIST FROM FUSED RANKING **
    chunk_lookup = {c["chunk_id"]: c for c in dense_chunks}
    for idx in bm25_top_indices:
        chunk = all_chunks[idx]
        if chunk["chunk_id"] not in chunk_lookup:
            chunk_lookup[chunk["chunk_id"]] = chunk

    final_chunks = []
    for chunk_id, rrf_score in fused_ranking[:k]:
        if chunk_id in chunk_lookup:
            chunk = chunk_lookup[chunk_id].copy()
            chunk["rrf_score"] = round(rrf_score, 6)
            chunk["score"]     = round(rrf_score, 6)  # **OVERRIDE FOR CONSISTENCY**
            final_chunks.append(chunk)

    return final_chunks[:k]


# ══════════════════════════════════════════════════════════
# BUILD BM25 INDEXES — RUN ONCE, REUSE FOR ALL HYBRID EVALS
# ══════════════════════════════════════════════════════════
print("Building BM25 indexes...")
bm25_recursive, chunks_recursive = build_bm25_index(col_recursive)
bm25_section,   chunks_section   = build_bm25_index(col_section)

In [ ]:
# ============================================================
# TASK 3 — GENERATION WITH GROUNDED PROMPTING
# RETRIEVAL: RERANKED | COLLECTION: SECTION_CHUNKS
# LLM: GPT-4O-MINI | CITATIONS: INLINE [SOURCE N]
# ============================================================

import json
import time
import re
from openai import OpenAI
from collections import Counter

# ── CONFIG ────────────────────────────────────────────────
QUERIES_PATH  = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/queries.json"
ANSWERS_PATH  = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/answers.json"
QRELS_PATH    = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/qrels.json"
OUTPUT_PATH   = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/task3_rag_output_1.json"
OPENAI_KEY    = userdata.get('OPENAI_API_KEY')
GEN_MODEL     = "gpt-4o-mini"           # **LLM FOR GENERATION**
TOP_K         = 5                       # **CHUNKS TO RETRIEVE**
TEST_LIMIT    = None                    # **SET TO None FOR FULL 496**

openai_client = OpenAI(api_key=OPENAI_KEY)  # **OPENAI CLIENT**


In [ ]:
# ══════════════════════════════════════════════════════════
# SYSTEM PROMPT
# ══════════════════════════════════════════════════════════
SYSTEM_PROMPT = """You are a precise academic research assistant. Your job is to answer questions about AI/ML research papers.

STRICT RULES:
1. Answer ONLY using information from the provided context chunks below.
2. You MUST cite sources inline using [Source N] notation after every claim.
3. If the context does not contain enough information to answer, respond with exactly: "I don't have enough information in the provided context to answer this question."
4. Do NOT use any prior knowledge or make assumptions beyond what is stated in the context.
5. Keep answers concise and factual.
6. If multiple sources support a claim, cite all of them e.g. [Source 1][Source 2].

FORMAT:
- Write in clear prose
- Use [Source N] immediately after each claim
- End with a "Sources Used:" section listing which sources were cited"""

In [ ]:
# ══════════════════════════════════════════════════════════
# HELPER: FORMAT CONTEXT CHUNKS
# ══════════════════════════════════════════════════════════
def format_context(chunks: list[dict]) -> str:
    """FORMAT RETRIEVED CHUNKS INTO NUMBERED SOURCE BLOCKS**"""
    context_blocks = []
    for i, chunk in enumerate(chunks, 1):
        first_line = chunk["text"].split("\n")[0].strip()
        title = first_line[:80] if first_line.startswith("#") else chunk["arxiv_id"]
        title = title.lstrip("#").strip()
        block = (
            f"[Source {i}: {chunk['arxiv_id']} — {title}]\n"
            f"{chunk['text'][:1000]}"  # **TRUNCATE LONG CHUNKS TO 1000 CHARS**
        )
        context_blocks.append(block)
    return "\n\n---\n\n".join(context_blocks)


# ══════════════════════════════════════════════════════════
# CORE FUNCTION: answer(query) → (str, list[chunk])
# ══════════════════════════════════════════════════════════
def answer(query: str, collection, strategy: str = "reranked") -> tuple[str, list[dict]]:
    """RETRIEVE CHUNKS THEN GENERATE GROUNDED ANSWER WITH CITATIONS**"""

    # STEP 1: RETRIEVE CHUNKS USING CHOSEN STRATEGY **
    if strategy == "reranked":
        chunks = retrieve_reranked(query, collection, k=TOP_K)
    elif strategy == "baseline":
        chunks = retrieve_baseline(query, collection, k=TOP_K)
    # elif strategy == "hyde":
    #     chunks = retrieve_hyde(query, collection, k=TOP_K)
    # elif strategy == "multiquery":
    #     chunks = retrieve_multiquery(query, collection, k=TOP_K)
    elif strategy == "hybrid":
        # Pass the global bm25_section and chunks_section for hybrid strategy
        chunks = retrieve_hybrid(query, collection, k=TOP_K, bm25_index=bm25_section, all_chunks=chunks_section)
    else:
        chunks = retrieve_baseline(query, collection, k=TOP_K)

    # STEP 2: FORMAT CONTEXT BEFORE QUESTION **
    context = format_context(chunks)

    # STEP 3: BUILD USER MESSAGE — CONTEXT FIRST THEN QUESTION **
    user_message = f"""CONTEXT:
{context}

---

QUESTION: {query}

Remember: Answer only from the context above. Cite every claim with [Source N]."""

    # STEP 4: CALL GPT WITH RETRY **
    for attempt in range(3):
        try:
            response = openai_client.chat.completions.create(
                model=GEN_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_message}
                ],
                temperature=0.0,    # **TEMPERATURE 0 FOR DETERMINISTIC FACTUAL ANSWERS**
                max_tokens=500
            )
            answer_text = response.choices[0].message.content.strip()
            return answer_text, chunks

        except Exception as e:
            print(f"  ⚠️ Attempt {attempt+1}/3 failed: {e}")
            if attempt < 2:
                time.sleep(3)
            else:
                return "Generation failed due to API error.", chunks

    return "Generation failed.", chunks


# ══════════════════════════════════════════════════════════
# BATCH GENERATION
# ══════════════════════════════════════════════════════════
def run_generation(queries, answers_gt, collection,
                   strategy="reranked", limit=None) -> list[dict]:
    """RUN GENERATION OVER ALL QUERIES AND COLLECT RESULTS FOR RAGAS**"""
    results      = []
    query_items  = list(queries.items())
    if limit:
        query_items = query_items[:limit]

    print(f"Running generation: {len(query_items)} queries | "
          f"strategy={strategy} | collection={collection.name}\n")

    for i, (qid, q_data) in enumerate(query_items):
        query_text   = q_data["query"]
        modality     = q_data.get("source", "text")    # **TEXT, TEXT-TABLE, TEXT-IMAGE**
        gt_answer    = answers_gt.get(qid, "")
        gt_answer    = gt_answer if isinstance(gt_answer, str) else str(gt_answer)

        # GENERATE ANSWER **
        answer_text, chunks = answer(query_text, collection, strategy=strategy)

        results.append({
            "query_id":         qid,
            "query":            query_text,
            "answer":           answer_text,
            "ground_truth":     gt_answer,
            "modality":         modality,
            "contexts":         [c["text"] for c in chunks],   # **FOR RAGAS**
            "retrieved_chunks": [
                {
                    "chunk_id": c["chunk_id"],
                    "arxiv_id": c["arxiv_id"],
                    "text":     c["text"],
                    "score":    c["score"],
                }
                for c in chunks
            ],
        })

        # PRINT PROGRESS EVERY 10 QUERIES **
        if (i + 1) % 10 == 0:
            print(f"  ✅ {i+1}/{len(query_items)} queries done")

    return results


# ══════════════════════════════════════════════════════════
# DISPLAY HELPER
# ══════════════════════════════════════════════════════════
def display_sample(result: dict):
    """PRETTY PRINT ONE RESULT FOR MANUAL INSPECTION**"""
    print("=" * 60)
    print(f"QUERY:    {result['query']}")
    print(f"MODALITY: {result['modality']}")
    print("-" * 60)
    print(f"ANSWER:\n{result['answer']}")
    print("-" * 60)
    print(f"GROUND TRUTH:\n{result['ground_truth']}")
    print("-" * 60)
    print("RETRIEVED SOURCES:")
    for i, c in enumerate(result["retrieved_chunks"], 1):
        print(f"  [Source {i}] {c['arxiv_id']} | score: {c['score']:.4f}")
        print(f"             {c['text'][:120]}...")
    print("=" * 60)

In [ ]:
# ══════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════
import numpy as np

strategy = "baseline"

# LOAD FILES **
print("Loading benchmark files...")
with open(QUERIES_PATH, "r") as f:
    queries = json.load(f)
with open(ANSWERS_PATH, "r") as f:
    answers_gt = json.load(f)

print(f"Queries: {len(queries)} | Answers: {len(answers_gt)}\n")

# ── SANITY CHECK: 25 QUERIES BEFORE FULL RUN ──────────────
print("SANITY CHECK — 25 sample queries")
print("=" * 60)
for qid, q_data in list(queries.items())[:25]:
    ans, chunks = answer(q_data["query"], col_section, strategy=strategy)
    print(f"\nQ: {q_data['query']}")
    print(f"A: {ans[:400]}")
    print(f"Sources: {[c['arxiv_id'] for c in chunks]}\n")

# ── FULL GENERATION RUN ───────────────────────────────────
print("\nStarting generation run...")
results = run_generation(
    queries    = queries,
    answers_gt = answers_gt,
    collection = col_section,
    strategy   = strategy,
    limit      = TEST_LIMIT
)

# ── DISPLAY 25 SAMPLE RESULTS ─────────────────────────────
print("\n\nSAMPLE RESULTS:")
for r in results[:25]:
    display_sample(r)

# ── MODALITY BREAKDOWN ────────────────────────────────────
print("\nMODALITY BREAKDOWN:")
modality_counts = Counter(r["modality"] for r in results)
for modality, count in sorted(modality_counts.items()):
    print(f"  {modality:<25} {count} queries")

In [ ]:
# ── SAVE ──────────────────────────────────────────────────
output = {
    "task":       "3.evaluation",
    "model":      GEN_MODEL,
    "strategy":   "hybrid",
    "collection": "section_chunks",
    "top_k":      TOP_K,
    "total":      len(results),
    "results":    results,
}
with open(OUTPUT_PATH, "w") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f"\n Saved {len(results)} results to {OUTPUT_PATH}")